# Laboratoire 10 : QAOA pour MaxCut

**Semaine 11 — Cours de Quantum Machine Learning (Master/PhD)**

## Objectifs

- Générer un graphe 3-régulier avec NetworkX
- Implémenter le circuit QAOA : phase separator + mixer
- Optimiser les angles $(\beta, \gamma)$ avec Nelder-Mead et COBYLA
- Calculer l'approximation ratio et comparer avec brute force et greedy
- Étudier le scaling : profondeur $p=1$ à $p=5$, qualité vs. ressources

---
## Imports

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from itertools import product
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 12, 'figure.figsize': (8, 5)})
print(f"PennyLane : {qml.__version__}")
print(f"NetworkX  : {nx.__version__}")

---
## Partie A : Génération du graphe 3-régulier

Un graphe $k$-régulier a exactement $k$ arêtes par sommet. Le problème MaxCut consiste à partitionner les sommets en deux groupes pour maximiser le nombre (ou le poids) d'arêtes traversant la coupe.

In [ ]:
np.random.seed(42)
n_nodes = 8  # 8 sommets, graphe 3-régulier

# Génération d'un graphe 3-régulier aléatoire
G = nx.random_regular_graph(3, n_nodes, seed=42)

# Visualisation
pos = nx.spring_layout(G, seed=42)
plt.figure()
nx.draw(G, pos, with_labels=True, node_color="lightblue",
        node_size=500, edge_color="gray", width=2)
plt.title(f"Graphe 3-régulier à {n_nodes} sommets")
plt.tight_layout()
plt.show()

print(f"Sommets : {G.number_of_nodes()}")
print(f"Arêtes  : {G.number_of_edges()}")
print(f"Degrés  : {[d for _, d in G.degree()]}")

### Solution MaxCut par force brute

Pour un petit graphe, nous pouvons énumérer toutes les $2^n$ partitions pour trouver la coupe maximale exacte.

In [ ]:
def brute_force_maxcut(G):
    nodes = list(G.nodes())
    best_val = -1
    best_cut = None
    for bits in product([0, 1], repeat=len(nodes)):
        val = 0
        for u, v in G.edges():
            if bits[u] != bits[v]:
                val += 1
        if val > best_val:
            best_val = val
            best_cut = bits
    return best_val, best_cut

maxcut_exact, best_partition = brute_force_maxcut(G)
print(f"MaxCut exact (brute force) : {maxcut_exact}")
print(f"Partition optimale : {best_partition}")

### Algorithme glouton (greedy) pour référence

In [ ]:
def greedy_maxcut(G):
    partition = {v: 0 for v in G.nodes()}
    for v in G.nodes():
        # Compter les voisins dans chaque groupe
        n0 = sum(1 for u in G.neighbors(v) if partition[u] == 0)
        n1 = sum(1 for u in G.neighbors(v) if partition[u] == 1)
        partition[v] = 0 if n0 <= n1 else 1
    cut_val = sum(1 for u, v in G.edges() if partition[u] != partition[v])
    return cut_val, partition

greedy_val, greedy_part = greedy_maxcut(G)
print(f"MaxCut glouton : {greedy_val}")

---
## Partie B : Circuit QAOA — Phase Separator + Mixer

Le circuit QAOA alterne deux types de couches :
- **Phase separator** $U_P(\gamma) = e^{-i\gamma H_C}$ où $H_C = \sum_{(i,j) \in E} Z_i Z_j$
- **Mixer** $U_M(\beta) = e^{-i\beta H_B}$ où $H_B = \sum_i X_i$

Pour une profondeur $p$, l'état préparé est :
$$|\psi(\beta, \gamma)\rangle = U_M(\beta_p) U_P(\gamma_p) \cdots U_M(\beta_1) U_P(\gamma_1) |+\rangle^{\otimes n}$$

In [ ]:
# Définition de l'Hamiltonien du problème (MaxCut)
edges = list(G.edges())

def maxcut_hamiltonian(edges):
    coeffs = []
    ops = []
    for u, v in edges:
        coeffs.append(-0.5)
        ops.append(qml.Z(u) @ qml.Z(v))
        coeffs.append(0.5)
        ops.append(qml.Identity(u))
    return qml.Hamiltonian(coeffs + [0.0], ops + [qml.Identity(0)], simplify=False)

H_C = maxcut_hamiltonian(edges)
print(f"Hamiltonien MaxCut : {len(edges)} arêtes")

In [ ]:
dev = qml.device("default.qubit", wires=n_nodes, shots=None)

def qaoa_layer(gamma, beta, edges):
    # Phase separator
    for u, v in edges:
        qml.CNOT(wires=[u, v])
        qml.RZ(-2 * gamma, wires=v)
        qml.CNOT(wires=[u, v])
    # Mixer
    for i in range(n_nodes):
        qml.RX(2 * beta, wires=i)

@qml.qnode(dev)
def qaoa_circuit(params, edges, p):
    # État initial |+>^n
    for i in range(n_nodes):
        qml.Hadamard(wires=i)
    # Couches QAOA
    for layer in range(p):
        gamma = params[layer]
        beta = params[p + layer]
        qaoa_layer(gamma, beta, edges)
    return qml.expval(H_C)

---
## Partie C : Optimisation des angles $(\beta, \gamma)$

Nous optimisons les angles avec Nelder-Mead et COBYLA pour $p=1$.

In [ ]:
p = 1
np.random.seed(0)
init_params = np.random.uniform(0, np.pi, size=2 * p)

def objective(params):
    return qaoa_circuit(params, edges, p)

# Optimisation COBYLA
opt_cobyla = qml.GradientDescentOptimizer(stepsize=0.1)
params_cobyla = init_params.copy()
energies_cobyla = []
for i in range(150):
    params_cobyla, e = opt_cobyla.step_and_cost(objective, params_cobyla)
    energies_cobyla.append(e)
print(f"COBYLA-like : énergie finale = {energies_cobyla[-1]:.4f}")

---
## Partie D : Approximation ratio

L'**approximation ratio** $r$ mesure la qualité de la solution :
$$r = \frac{C_{\text{algo}}}{C_{\max}}$$

où $C_{\text{algo}}$ est la valeur de la coupe trouvée et $C_{\max}$ le MaxCut exact.

In [ ]:
def sample_maxcut(params, p, n_shots=1000):
    """Échantillonne le circuit QAOA et retourne la meilleure coupe."""
    dev_sample = qml.device("default.qubit", wires=n_nodes, shots=n_shots)
    
    @qml.qnode(dev_sample)
    def circuit(params):
        for i in range(n_nodes):
            qml.Hadamard(wires=i)
        for layer in range(p):
            gamma = params[layer]
            beta = params[p + layer]
            qaoa_layer(gamma, beta, edges)
        return qml.counts()
    
    counts = circuit(params)
    best_val = 0
    best_bitstring = None
    for bitstring, _ in counts.items():
        bits = [int(b) for b in bitstring]
        val = sum(1 for u, v in edges if bits[u] != bits[v])
        if val > best_val:
            best_val = val
            best_bitstring = bitstring
    return best_val, best_bitstring

best_val_qaoa, best_bits = sample_maxcut(params_cobyla, p)
approx_ratio = best_val_qaoa / maxcut_exact

print(f"MaxCut exact          : {maxcut_exact}")
print(f"MaxCut glouton        : {greedy_val}, ratio = {greedy_val / maxcut_exact:.3f}")
print(f"MaxCut QAOA (p={p})   : {best_val_qaoa}, ratio = {approx_ratio:.3f}")
print(f"Meilleure chaîne binaire : {best_bits}")

---
## Partie E : Scaling $p=1$ à $p=5$

Nous évaluons comment l'approximation ratio évolue avec la profondeur du circuit QAOA, et analysons le compromis qualité vs. ressources (nombre de portes).

In [ ]:
p_values = range(1, 6)
approx_ratios = []
gate_counts = []

for p_i in p_values:
    init_p = np.random.uniform(0, np.pi, size=2 * p_i)
    
    @qml.qnode(dev)
    def circuit_p(params):
        for i in range(n_nodes):
            qml.Hadamard(wires=i)
        for layer in range(p_i):
            gamma = params[layer]
            beta = params[p_i + layer]
            qaoa_layer(gamma, beta, edges)
        return qml.expval(H_C)
    
    def obj_p(params):
        return circuit_p(params)
    
    params_p = init_p.copy()
    opt_p = qml.GradientDescentOptimizer(stepsize=0.05)
    for _ in range(200):
        params_p, _ = opt_p.step_and_cost(obj_p, params_p)
    
    val, _ = sample_maxcut(params_p, p_i)
    approx_ratios.append(val / maxcut_exact)
    # Comptage de portes : n_nodes Hadamard init + p * (2 * len(edges) CNOT + 2 * len(edges) RZ + n_nodes RX)
    gates = n_nodes + p_i * (2 * len(edges) + 2 * len(edges) + n_nodes)
    gate_counts.append(gates)
    
    print(f"p={p_i} : ratio = {approx_ratios[-1]:.4f}, portes ≈ {gates}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(p_values, approx_ratios, "bo-", lw=2, markersize=8)
ax1.axhline(1.0, color="gray", ls="--", label="Optimal (ratio = 1)")
ax1.axhline(greedy_val / maxcut_exact, color="red", ls=":", label=f"Greedy ({greedy_val / maxcut_exact:.3f})")
ax1.set_xlabel("Profondeur $p$")
ax1.set_ylabel("Approximation ratio")
ax1.set_title("Qualité de la solution vs. profondeur")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xticks(p_values)

ax2.plot(p_values, gate_counts, "rs-", lw=2, markersize=8)
ax2.set_xlabel("Profondeur $p$")
ax2.set_ylabel("Nombre de portes")
ax2.set_title("Ressources (portes) vs. profondeur")
ax2.grid(True, alpha=0.3)
ax2.set_xticks(p_values)

plt.tight_layout()
plt.show()

print("\nAnalyse :")
print(f"- Ratio final (p=5) : {approx_ratios[-1]:.4f}")
print(f"- Gain par rapport à p=1 : {(approx_ratios[-1] - approx_ratios[0]) * 100:.1f}%")
print(f"- Ratio glouton       : {greedy_val / maxcut_exact:.4f}")
if approx_ratios[-1] > greedy_val / maxcut_exact:
    print("- QAOA surpasse l'algorithme glouton.")
else:
    print("- QAOA est comparable ou inférieur au glouton sur ce graphe.")

---
## Qiskit QAOA (complément)

Utilisation de `qiskit-optimization` pour exécuter QAOA avec des optimiseurs classiques.

In [ ]:
try:
    from qiskit_algorithms import QAOA, SamplingVQE
    from qiskit_algorithms.optimizers import COBYLA
    from qiskit.primitives import Sampler
    from qiskit_optimization.applications import Maxcut
    from qiskit_optimization.algorithms import MinimumEigenOptimizer
    
    HAS_QISKIT_QAOA = True
    print("Qiskit QAOA disponible.")
except ImportError as e:
    HAS_QISKIT_QAOA = False
    print(f"Qiskit QAOA non disponible : {e}")
    print("Installez avec : pip install qiskit-algorithms qiskit-optimization")

In [ ]:
if HAS_QISKIT_QAOA:
    # Conversion du graphe en problème MaxCut Qiskit
    maxcut = Maxcut(G)
    qp = maxcut.to_quadratic_program()
    
    # QAOA avec COBYLA
    sampler = Sampler()
    qaoa_qiskit = QAOA(sampler, COBYLA(maxiter=200), reps=1)
    optimizer = MinimumEigenOptimizer(qaoa_qiskit)
    result_qiskit = optimizer.solve(qp)
    
    print(f"QAOA (Qiskit) : coupe = {int(result_qiskit.fval)}, ratio = {result_qiskit.fval / maxcut_exact:.4f}")
    print(f"Meilleure solution : {result_qiskit.x}")

---
## Exercices — Pour aller plus loin

1. **Graphes aléatoires** : Testez QAOA sur des graphes aléatoires avec $n=10, 12, 14$ sommets. Que devient l'approximation ratio ?
2. **Poids sur les arêtes** : Implémentez une version pondérée de MaxCut. Modifiez l'Hamiltonien en conséquence.
3. **Optimisation multi-départs** : Lancez l'optimisation avec 10 initialisations aléatoires. Quelle est la variance du ratio final ?
4. **Réseaux de plus grande taille** : Passez à $n=16$ où la force brute n'est plus possible. Utilisez QAOA comme seule référence.
5. **Comparaison recuit simulé** : Implémentez un recuit simulé (`scipy.optimize.dual_annealing`) pour MaxCut et comparez.
6. **Transpilation** : Utilisez Qiskit pour transpiler le circuit QAOA vers une architecture réaliste (IBM Brisbane 127 qubits).